# Agentic RAG

**PDF → Chunk → Embed → ChromaDB → Retrieve → Generate**

An end-to-end Retrieval-Augmented Generation pipeline. It turns a folder of PDFs into a searchable vector index, retrieves the passages most relevant to a question, and asks a language model to answer using only that retrieved evidence.

## What this notebook covers

| Step | What happens |
|---|---|
| 1. Load | Read every PDF page with `PyPDFLoader` |
| 2. Chunk | Split pages into overlapping 1,000-character chunks |
| 3. Embed | Encode chunks with `all-MiniLM-L6-v2` (384 dimensions) |
| 4. Store | Persist vectors in a local ChromaDB collection |
| 5. Index | Upsert chunks with stable IDs — safe to re-run |
| 6. Retrieve | Find the nearest chunks by cosine similarity |
| 7. Generate | Answer with Gemini, grounded in the retrieved context |

## Before you run

- Put your PDFs in `src/data/text_files/`.
- Add `GOOGLE_API_KEY=...` to `.env` for the answer-generation step.
- The first run downloads the embedding model and builds the index; later runs reuse it.
- Retrieval works without an API key. Only the final generation step needs one.

## Setup

Import the libraries used throughout the notebook and load environment variables from `.env`.

In [15]:
import hashlib
import os
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

load_dotenv()

True

### Configuration

All tunable values live in one place. Paths are resolved from the nearest `pyproject.toml`, so the notebook works whether Jupyter starts in the repository root or in `src/notebook/`.

In [16]:
def find_project_root(start: Path | None = None) -> Path:
    """Return the nearest directory containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "src" / "data"
PDF_DIR = DATA_DIR / "text_files"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"

COLLECTION_NAME = "pdf_documents"
CHUNK_SIZE = 1_000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "gemini-3.5-flash"
LLM_TEMPERATURE = 0.1
LLM_MAX_TOKENS = 1_024
TOP_K = 5
SCORE_THRESHOLD = 0.0
RESET_COLLECTION = False

PDF_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"PDFs         : {PDF_DIR}")
print(f"Vector store : {VECTOR_STORE_DIR}")

Project root : C:\Users\risha\learning\agentic-rag
PDFs         : C:\Users\risha\learning\agentic-rag\src\data\text_files
Vector store : C:\Users\risha\learning\agentic-rag\src\data\vector_store


---

## Step 1 — Load PDFs

Recursively scan the PDF directory with `PyPDFLoader`. Each page becomes a LangChain `Document` tagged with its source filename, which lets us trace answers back to a document later.

In [17]:
def load_pdfs(pdf_directory: Path) -> list:
    """Load every page from every PDF under pdf_directory."""
    documents = []
    for pdf_file in sorted(pdf_directory.glob("**/*.pdf")):
        pages = PyPDFLoader(str(pdf_file)).load()
        for page in pages:
            page.metadata["source_file"] = pdf_file.name
            page.metadata["file_type"] = "pdf"
        documents.extend(pages)
    return documents


all_pdf_documents = load_pdfs(PDF_DIR)

if not all_pdf_documents:
    raise FileNotFoundError(f"No PDFs found in {PDF_DIR}")

source_count = len({doc.metadata["source_file"] for doc in all_pdf_documents})
print(f"Loaded {len(all_pdf_documents)} pages from {source_count} PDF(s)")

Loaded 948 pages from 1 PDF(s)


## Step 2 — Chunk

PDF pages are too long to embed as a single vector, so split them into overlapping chunks. The overlap preserves sentences that straddle a chunk boundary.

- `CHUNK_SIZE` controls how much text each vector represents.
- `CHUNK_OVERLAP` prevents important context from being cut in half.

In [18]:
def split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Split documents into overlapping character chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documents)


chunks = split_documents(all_pdf_documents)

if not chunks:
    raise ValueError("No chunks were produced. Check that the PDFs contain extractable text.")

for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = index

average_length = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)
print(f"Split {len(all_pdf_documents)} pages into {len(chunks)} chunks")
print(f"Average chunk length: {average_length:.0f} characters")

Split 948 pages into 1427 chunks
Average chunk length: 696 characters


## Step 3 — Embed

Convert text to vectors with SentenceTransformers. `all-MiniLM-L6-v2` is small, fast, and produces 384-dimensional embeddings that work well for semantic search.

In [19]:
class EmbeddingManager:
    """Thin wrapper around a SentenceTransformer model."""

    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model = SentenceTransformer(model_name)

    @property
    def dimension(self) -> int:
        return self.model.get_embedding_dimension()

    def embed(self, texts: list[str]) -> np.ndarray:
        """Encode texts into a (len(texts), dimension) array."""
        return self.model.encode(texts, show_progress_bar=len(texts) > 1)


embedding_manager = EmbeddingManager()
print(f"Embedding model: {EMBEDDING_MODEL} ({embedding_manager.dimension} dimensions)")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3241.62it/s]


Embedding model: all-MiniLM-L6-v2 (384 dimensions)


## Step 4 — Vector Store

ChromaDB persists embeddings on disk so the index survives notebook restarts. The collection uses **cosine distance**, which is the usual choice for normalized sentence embeddings.

`VectorStore.sync` is idempotent:

1. Each chunk gets a stable ID derived from its source, page, and text.
2. Existing chunks are upserted instead of duplicated.
3. Entries that no longer exist in the corpus are removed.

> If you are upgrading an older index, set `RESET_COLLECTION = True` once. That deletes the old collection and recreates it with the cosine metric.

In [20]:
class VectorStore:
    """ChromaDB collection that stores chunks and their embeddings."""

    def __init__(
        self,
        collection_name: str = COLLECTION_NAME,
        persist_directory: Path = VECTOR_STORE_DIR,
    ):
        self.collection_name = collection_name
        self.persist_directory = Path(persist_directory)
        self.client = chromadb.PersistentClient(path=str(self.persist_directory))
        self.collection = self._get_or_create_collection()

    def _get_or_create_collection(self):
        return self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "hnsw:space": "cosine",
                "description": "PDF chunks for RAG",
            },
        )

    def reset(self) -> None:
        """Delete the collection and create an empty one."""
        self.client.delete_collection(self.collection_name)
        self.collection = self._get_or_create_collection()

    def sync(self, documents, embeddings: np.ndarray) -> None:
        """Upsert chunks with stable IDs and remove stale entries."""
        if len(documents) != len(embeddings):
            raise ValueError("documents and embeddings must have the same length")

        records = {}
        for doc, embedding in zip(documents, embeddings):
            key = f"{doc.metadata.get('source_file')}:{doc.metadata.get('page')}:{doc.page_content}"
            doc_id = hashlib.sha256(key.encode("utf-8")).hexdigest()[:32]
            metadata = {
                name: value
                for name, value in doc.metadata.items()
                if isinstance(value, (str, int, float, bool))
            }
            metadata["content_length"] = len(doc.page_content)
            records[doc_id] = {
                "embedding": embedding.tolist(),
                "metadata": metadata,
                "document": doc.page_content,
            }

        ids = list(records)
        existing = set(self.collection.get(include=[])["ids"])
        stale = existing.difference(ids)
        if stale:
            self.collection.delete(ids=list(stale))

        self.collection.upsert(
            ids=ids,
            embeddings=[records[doc_id]["embedding"] for doc_id in ids],
            metadatas=[records[doc_id]["metadata"] for doc_id in ids],
            documents=[records[doc_id]["document"] for doc_id in ids],
        )
        print(
            f"Indexed {len(ids)} chunks (removed {len(stale)} stale) — "
            f"collection now has {self.collection.count()} entries"
        )


vector_store = VectorStore()
print(f"Collection '{COLLECTION_NAME}' currently has {vector_store.collection.count()} entries")

space = (getattr(vector_store.collection, "configuration_json", {}) or {}).get("hnsw", {}).get("space")
if space and space != "cosine":
    print(
        f"Warning: this collection uses '{space}' distance. "
        "Set RESET_COLLECTION = True to rebuild it with cosine."
    )

Collection 'pdf_documents' currently has 4281 entries


## Step 5 — Index

Embed every chunk and write it to ChromaDB. Re-running this cell is safe: unchanged chunks are upserted, duplicates are avoided, and stale chunks are pruned.

In [21]:
if RESET_COLLECTION:
    vector_store.reset()
    print("Deleted the existing collection and created a fresh one.")

texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_manager.embed(texts)
vector_store.sync(chunks, embeddings)

Batches: 100%|██████████| 45/45 [02:27<00:00,  3.27s/it]


Indexed 1427 chunks (removed 4281 stale) — collection now has 1427 entries


## Step 6 — Retrieve

Embed the question with the same model, then ask ChromaDB for the nearest chunks. Cosine distance is converted back to a similarity score in the range `[-1, 1]`:

```text
similarity = 1 - cosine_distance
```

`score_threshold` filters out weak matches; exact duplicate chunks are skipped so the context stays diverse.

In [22]:
class RAGRetriever:
    """Semantic search over a VectorStore."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = TOP_K,
        score_threshold: float = SCORE_THRESHOLD,
    ) -> list[dict[str, Any]]:
        """Return up to top_k chunks with cosine similarity >= threshold."""
        total = self.vector_store.collection.count()
        if total == 0:
            return []

        query_embedding = self.embedding_manager.embed([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(top_k, total),
            include=["documents", "metadatas", "distances"],
        )

        retrieved = []
        seen_content = set()
        for doc_id, content, metadata, distance in zip(
            results["ids"][0],
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            if content in seen_content:
                continue

            score = 1.0 - float(distance)
            if score < score_threshold:
                continue

            seen_content.add(content)
            retrieved.append(
                {
                    "id": doc_id,
                    "content": content,
                    "metadata": metadata,
                    "score": score,
                    "rank": len(retrieved) + 1,
                }
            )

        return retrieved


rag_retriever = RAGRetriever(vector_store, embedding_manager)

## Step 7 — Generate

Configure Gemini and wrap retrieval plus generation in one function. The prompt instructs the model to:

- answer **only** from the retrieved context,
- admit when the context is insufficient,
- cite passages with `[1]`, `[2]`, and so on.

In [23]:
if not os.getenv("GOOGLE_API_KEY"):
    raise EnvironmentError(
        "GOOGLE_API_KEY is not set. Add it to .env before running the generation step."
    )

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
)

In [24]:
SYSTEM_PROMPT = """You are a careful research assistant.
Answer the question using only the context provided.
If the answer is not in the context, say that the context does not contain enough information.
Cite supporting passages with bracketed numbers such as [1] or [2]."""


def format_context(results: list[dict[str, Any]]) -> str:
    """Format retrieved chunks into numbered context blocks."""
    blocks = []
    for result in results:
        metadata = result["metadata"]
        source = metadata.get("source_file", "unknown")
        page = metadata.get("page")
        location = f"{source}, page {int(page) + 1}" if page is not None else source
        blocks.append(f"[{result['rank']}] {location}\n{result['content']}")
    return "\n\n---\n\n".join(blocks)


def message_text(message) -> str:
    """Extract plain text from a LangChain message, ignoring non-text blocks."""
    content = message.content
    if isinstance(content, str):
        return content
    if not content:
        return ""

    parts = []
    for block in content:
        if isinstance(block, str):
            parts.append(block)
        elif isinstance(block, dict) and block.get("type") == "text":
            parts.append(block.get("text", ""))
    return "".join(parts).strip()


def rag_answer(
    query: str,
    retriever: RAGRetriever,
    llm: ChatGoogleGenerativeAI,
    top_k: int = TOP_K,
) -> tuple[str, list[dict[str, Any]]]:
    """Retrieve evidence and generate a grounded answer."""
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return "No relevant context found to answer the question.", []

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(
            content=(
                f"Context:\n{format_context(results)}\n\n"
                f"Question: {query}\n\n"
                "Answer:"
            )
        ),
    ]
    response = llm.invoke(messages)
    return message_text(response), results

## Step 8 — Ask a question

Run a query end to end and print the answer with its supporting sources.

In [25]:
question = "Why is business analytics important?"
answer, sources = rag_answer(question, rag_retriever, llm, top_k=TOP_K)

print(f"Question: {question}\n")
print(answer)

print("\nSources")
for result in sources:
    metadata = result["metadata"]
    print(
        f"  [{result['rank']}] {metadata.get('source_file')} "
        f"— score {result['score']:.3f}"
    )

Question: Why is business analytics important?

Based on the provided context, business analytics (and the application of analytics) is important because it:

* **Improves decision-making:** It supports data analysis using statistics, math, and algorithmic models to help organizations make better, data-driven decisions in real time [1], [3].
* **Drives business performance:** Leveraging data results in improved efficiency, effectiveness, and a better bottom line [2].
* **Solves business problems and creates value:** It plays a critical role in attaining business value and addressing specific organizational challenges, such as campaign optimization and reducing hours spent on report preparation [2], [3].
* **Provides business insight from data:** It allows organizations to exploit large datasets for insight by applying business expertise to technical, mathematical, and statistical findings [4]. 
* **Plays a critical organizational role:** Analytics plays a critical part in modern organi

---

## Tuning and troubleshooting

| Symptom | Try |
|---|---|
| Answers miss relevant passages | Increase `TOP_K`, lower `SCORE_THRESHOLD`, or reduce `CHUNK_SIZE` |
| Answers are too long | Lower `LLM_MAX_TOKENS` or tighten the system prompt |
| Answers are not grounded | Keep `LLM_TEMPERATURE` between `0.0` and `0.2` |
| Index is out of date | Re-run Step 5 — it upserts changes and prunes stale chunks |
| Collection needs a clean rebuild | Set `RESET_COLLECTION = True`, re-run Step 5, then set it back to `False` |
| `GOOGLE_API_KEY` error | Add the key to `.env`; retrieval still works without it |

### Where to go next

- Add a reranker to improve precision after retrieval.
- Return page-level citations instead of chunk-level scores.
- Replace ChromaDB with FAISS or a managed vector database.
- Wrap the pipeline in an agent that can decide when to retrieve.